# HRAF Misfortune Multi-Label Classification Training

This notebook trains a RoBERTa-based multi-label classifier on the **tiered quality-filtered dataset** (5000 passages).

## Key Configuration
- Uses tiered dataset (2000 high-quality + 3000 medium-quality passages)
- Asymmetric Weighted Focal Loss (gamma=4.0, pos_weight_multiplier=3.0)
- RoBERTa-base encoder with 3-layer classifiers
- Early stopping with patience=4

In [ ]:
# ============================================================================
# CELL 1: IMPORTS AND SETUP
# ============================================================================
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    AutoTokenizer,
    PreTrainedModel,
    PretrainedConfig,
    AutoModel,
    AutoConfig,
    EarlyStoppingCallback,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from typing import Optional, Dict, List, Tuple
import warnings
import os
import json
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Set device - prefer MPS on Mac, then CUDA, then CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION
# ============================================================================

# Model configuration - matches best performing model
CONFIG = {
    # Base model
    "base_model": "roberta-base",
    
    # Architecture - flat model (no hierarchy)
    "use_hierarchy": False,
    "gated_hierarchy": False,
    "gate_threshold": 0.5,
    "predict_main_labels": False,  # No EVENT/CAUSE/ACTION main labels
    
    # Model dimensions
    "hidden_size": 768,
    "hierarchical_hidden_size": 768,
    "num_hidden_layers": 3,
    
    # Regularization
    "dropout": 0.15,
    "attention_dropout": 0.1,
    
    # Loss configuration - KEY FOR PERFORMANCE
    "use_weighted_loss": True,
    "use_focal_loss": True,
    "focal_gamma": 4.0,
    "pos_weight_multiplier": 3.0,
    
    # Training parameters
    "teacher_forcing_ratio": 0.0,
    "num_epochs": 13,
    "batch_size": 12,
    "gradient_accumulation_steps": 1,
    "learning_rate": 2e-05,
    "warmup_steps": 500,
    "weight_decay": 0.02,
    "max_length": 512,
    "label_smoothing": 0.0,
    
    # Early stopping
    "use_early_stopping": True,
    "early_stopping_patience": 4,
    "early_stopping_threshold": 0.001,
    
    # Data splits
    "test_size": 0.2,
    "validation_size": 0.1,
    "random_seed": 42,
    "stratify_by": "Illness",
}

# Label columns - simple names (no prefixes)
LABEL_COLUMNS = [
    "Illness",
    "Accident",
    "Other",
    "Material_Physical",
    "Spirits_Gods",
    "Witchcraft_Sorcery",
    "Rule_Violation_Taboo",
    "Physical_Material",
    "Technical_Specialist",
    "Divination",
    "Shaman_Medium_Healer",
    "Priest_High_Religion"
]

# Data path - TIERED DATASET
DATA_PATH = "data/objects/tiered/tiered_scored_embedded_cleaned_raw__Altogether_Dataset_RACoded_Combined_20251014_091039/data.xlsx"

print("Configuration loaded!")
print(f"Number of labels: {len(LABEL_COLUMNS)}")
print(f"Loss: Asymmetric Weighted Focal Loss (gamma={CONFIG['focal_gamma']}, pos_multiplier={CONFIG['pos_weight_multiplier']})")

In [ ]:
# ============================================================================
# CELL 3: MODEL DEFINITION
# ============================================================================

class ConfigurableHierarchicalConfig(PretrainedConfig):
    """Configuration for configurable hierarchical model"""
    model_type = "configurable_hierarchical"

    def __init__(
        self,
        base_model="roberta-base",
        use_hierarchy=False,
        gated_hierarchy=False,
        gate_threshold=0.5,
        hidden_size=768,
        hierarchical_hidden_size=768,
        num_hidden_layers=3,
        dropout=0.15,
        attention_dropout=0.1,
        use_weighted_loss=True,
        use_focal_loss=True,
        focal_gamma=4.0,
        teacher_forcing_ratio=0.0,
        predict_main_labels=False,
        num_main_labels=0,
        num_event_labels=3,
        num_cause_labels=4,
        num_action_labels=5,
        total_labels=12,
        label_indices=None,
        label_names=None,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.base_model = base_model
        self.use_hierarchy = use_hierarchy
        self.gated_hierarchy = gated_hierarchy
        self.gate_threshold = gate_threshold
        self.hidden_size = hidden_size
        self.hierarchical_hidden_size = hierarchical_hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.dropout = dropout
        self.attention_dropout = attention_dropout
        self.use_weighted_loss = use_weighted_loss
        self.use_focal_loss = use_focal_loss
        self.focal_gamma = focal_gamma
        self.teacher_forcing_ratio = teacher_forcing_ratio
        self.predict_main_labels = predict_main_labels
        self.num_main_labels = num_main_labels
        self.num_event_labels = num_event_labels
        self.num_cause_labels = num_cause_labels
        self.num_action_labels = num_action_labels
        self.total_labels = total_labels
        self.label_indices = label_indices or {}
        self.label_names = label_names or []


class ConfigurableHierarchicalModel(PreTrainedModel):
    """Configurable multi-label classifier"""
    config_class = ConfigurableHierarchicalConfig
    base_model_prefix = "configurable_hierarchical"
    supports_gradient_checkpointing = True

    def __init__(self, config: ConfigurableHierarchicalConfig):
        super().__init__(config)
        self.config = config
        self.encoder = AutoModel.from_pretrained(config.base_model)

        if hasattr(config, 'attention_dropout') and config.attention_dropout > 0:
            self.encoder.config.attention_probs_dropout_prob = config.attention_dropout

        # No main labels for flat model
        self.main_classifier = None
        hierarchical_input_size = config.hidden_size

        # Build sublabel classifiers
        self.event_classifier = self._build_sublabel_classifier(
            hierarchical_input_size, config.num_event_labels,
            config.hierarchical_hidden_size, config.num_hidden_layers, config.dropout
        )
        self.cause_classifier = self._build_sublabel_classifier(
            hierarchical_input_size, config.num_cause_labels,
            config.hierarchical_hidden_size, config.num_hidden_layers, config.dropout
        )
        self.action_classifier = self._build_sublabel_classifier(
            hierarchical_input_size, config.num_action_labels,
            config.hierarchical_hidden_size, config.num_hidden_layers, config.dropout
        )

        self.use_hierarchy = config.use_hierarchy
        self.gated_hierarchy = config.gated_hierarchy
        self.gate_threshold = config.gate_threshold
        self.post_init()

    def _build_sublabel_classifier(self, input_size, output_size, hidden_size, num_layers, dropout):
        if output_size == 0:
            return None
        layers = []
        for i in range(num_layers):
            if i == 0:
                layers.append(nn.Linear(input_size, hidden_size))
            else:
                layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(hidden_size, output_size))
        return nn.Sequential(*layers)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        teacher_forcing=False,
        return_dict=None,
        **kwargs
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        encoder_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        pooled_output = encoder_outputs.last_hidden_state[:, 0]

        # No main labels - use pooled output directly
        hierarchical_input = pooled_output

        # Get sublabel predictions
        event_logits = self.event_classifier(hierarchical_input) if self.event_classifier else torch.zeros(pooled_output.shape[0], 0).to(pooled_output.device)
        cause_logits = self.cause_classifier(hierarchical_input) if self.cause_classifier else torch.zeros(pooled_output.shape[0], 0).to(pooled_output.device)
        action_logits = self.action_classifier(hierarchical_input) if self.action_classifier else torch.zeros(pooled_output.shape[0], 0).to(pooled_output.device)

        # Concatenate all logits
        logits = torch.cat([event_logits, cause_logits, action_logits], dim=1)

        loss = None
        if labels is not None:
            if self.config.use_focal_loss:
                loss = self._focal_loss(logits, labels.float(), gamma=self.config.focal_gamma)
            else:
                loss_fct = nn.BCEWithLogitsLoss()
                loss = loss_fct(logits, labels.float())

        if not return_dict:
            output = (logits,) + encoder_outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=encoder_outputs.hidden_states,
            attentions=encoder_outputs.attentions,
        )

    def _focal_loss(self, logits, targets, gamma=2.0):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probas = torch.sigmoid(logits)
        focal_weight = torch.where(targets == 1, (1 - probas) ** gamma, probas ** gamma)
        focal_loss = focal_weight * bce_loss
        return focal_loss.mean()


# Register model
AutoConfig.register("configurable_hierarchical", ConfigurableHierarchicalConfig)
AutoModel.register(ConfigurableHierarchicalConfig, ConfigurableHierarchicalModel)

print("Model classes defined and registered!")

In [ ]:
# ============================================================================
# CELL 4: CUSTOM TRAINER WITH ASYMMETRIC WEIGHTED FOCAL LOSS
# ============================================================================

class HierarchicalTrainer(Trainer):
    """Custom trainer with asymmetric weighted focal loss"""

    def __init__(self, pos_weights=None, neg_weights=None, teacher_forcing_ratio=0.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weights = pos_weights
        self.neg_weights = neg_weights
        self.teacher_forcing_ratio = teacher_forcing_ratio

        if pos_weights is not None:
            self.pos_weights = pos_weights.to(self.args.device)
        if neg_weights is not None:
            self.neg_weights = neg_weights.to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        """Custom loss computation with weighted focal loss"""
        labels = inputs.pop("labels")

        use_teacher_forcing = model.training and (torch.rand(1).item() < self.teacher_forcing_ratio)

        outputs = model(
            **inputs,
            labels=None,
            teacher_forcing=use_teacher_forcing
        )

        logits = outputs.logits

        # Asymmetric Weighted Focal Loss
        if self.pos_weights is not None and model.config.use_weighted_loss:
            gamma = model.config.focal_gamma if model.config.use_focal_loss else 0.0

            bce_loss = nn.functional.binary_cross_entropy_with_logits(
                logits, labels.float(), reduction='none'
            )

            probs = torch.sigmoid(logits)
            if gamma > 0:
                focal_weight = torch.where(
                    labels == 1,
                    (1 - probs) ** gamma,
                    probs ** gamma
                )
            else:
                focal_weight = torch.ones_like(probs)

            pos_weights_expanded = self.pos_weights.unsqueeze(0).expand_as(logits)
            neg_weights_expanded = self.neg_weights.unsqueeze(0).expand_as(logits)

            class_weights = torch.where(
                labels == 1,
                pos_weights_expanded,
                neg_weights_expanded
            )

            weighted_focal_loss = focal_weight * bce_loss * class_weights
            loss = weighted_focal_loss.mean()
        else:
            if model.config.use_focal_loss:
                loss = self._focal_loss(logits, labels.float(), gamma=model.config.focal_gamma)
            else:
                loss_fct = nn.BCEWithLogitsLoss()
                loss = loss_fct(logits, labels.float())

        return (loss, outputs) if return_outputs else loss

    def _focal_loss(self, logits, targets, gamma=2.0):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probas = torch.sigmoid(logits)
        focal_weight = torch.where(targets == 1, (1 - probas) ** gamma, probas ** gamma)
        return (focal_weight * bce_loss).mean()

print("Custom trainer defined!")

In [ ]:
# ============================================================================
# CELL 5: DATA LOADING
# ============================================================================

def load_and_clean_data(data_path: str, label_columns: List[str]) -> pd.DataFrame:
    """Load and clean the tiered dataset"""
    print(f"Loading data from: {data_path}")
    df = pd.read_excel(data_path)
    print(f"Initial passages: {len(df)}")

    # Ensure Passage column exists
    if 'Passage' not in df.columns:
        raise ValueError("Data must have 'Passage' column")

    # Remove passages with missing text
    df_clean = df[df['Passage'].notna() & (df['Passage'].str.strip() != '')].copy()
    print(f"After removing empty passages: {len(df_clean)}")

    # Ensure all label columns exist and are binary
    for label in label_columns:
        if label not in df_clean.columns:
            print(f"Warning: {label} not found, adding as zeros")
            df_clean[label] = 0
        else:
            df_clean[label] = df_clean[label].fillna(0).astype(int)

    # Print label distribution
    print("\nLabel distribution:")
    for label in label_columns:
        count = df_clean[label].sum()
        pct = count / len(df_clean) * 100
        print(f"  {label}: {count} ({pct:.1f}%)")

    return df_clean.reset_index(drop=True)


def calculate_class_weights(
    df: pd.DataFrame,
    label_columns: List[str],
    max_weight: float = 10.0,
    pos_weight_multiplier: float = 1.0
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Calculate asymmetric class weights"""
    pos_weights = []
    neg_weights = []

    print(f"\nClass weights (pos_multiplier={pos_weight_multiplier}x):")
    for label in label_columns:
        pos_count = df[label].sum()
        neg_count = len(df) - pos_count
        total = len(df)

        if pos_count > 0:
            pos_freq = pos_count / total
            neg_freq = neg_count / total
            raw_pos_weight = neg_freq / pos_freq
            pos_weight = min(np.sqrt(raw_pos_weight) * pos_weight_multiplier, max_weight)
            neg_weight = 1.0
        else:
            pos_weight = 1.0
            neg_weight = 1.0

        pos_weights.append(pos_weight)
        neg_weights.append(neg_weight)
        print(f"  {label}: {pos_count} positive ({pos_count/total*100:.1f}%) -> pos_weight={pos_weight:.2f}")

    return (
        torch.tensor(pos_weights, dtype=torch.float32),
        torch.tensor(neg_weights, dtype=torch.float32)
    )

# Load data
df = load_and_clean_data(DATA_PATH, LABEL_COLUMNS)

# Calculate class weights
pos_weights, neg_weights = calculate_class_weights(
    df, LABEL_COLUMNS,
    max_weight=10.0,
    pos_weight_multiplier=CONFIG["pos_weight_multiplier"]
)

In [ ]:
# ============================================================================
# CELL 6: PREPARE DATASETS
# ============================================================================

# Initialize tokenizer
print(f"Loading tokenizer: {CONFIG['base_model']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])

# Split data
train_val_df, test_df = train_test_split(
    df,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_seed"],
    stratify=df[CONFIG["stratify_by"]] if CONFIG["stratify_by"] else None
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=CONFIG["validation_size"] / (1 - CONFIG["test_size"]),
    random_state=CONFIG["random_seed"],
    stratify=train_val_df[CONFIG["stratify_by"]] if CONFIG["stratify_by"] else None
)

print(f"\nData splits:")
print(f"  Training: {len(train_df)} passages")
print(f"  Validation: {len(val_df)} passages")
print(f"  Test: {len(test_df)} passages")

# Convert to HuggingFace datasets
def tokenize_function(examples):
    return tokenizer(
        examples['Passage'],
        padding='max_length',
        truncation=True,
        max_length=CONFIG["max_length"]
    )

def prepare_labels(examples):
    labels = []
    batch_size = len(examples[LABEL_COLUMNS[0]])
    for i in range(batch_size):
        label_vector = [examples[col][i] for col in LABEL_COLUMNS]
        labels.append(label_vector)
    examples['labels'] = labels
    return examples

# Create datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Tokenize
print("\nTokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Prepare labels
train_dataset = train_dataset.map(prepare_labels, batched=True)
val_dataset = val_dataset.map(prepare_labels, batched=True)
test_dataset = test_dataset.map(prepare_labels, batched=True)

# Remove unnecessary columns
columns_to_remove = ['Passage'] + LABEL_COLUMNS + [c for c in train_dataset.column_names if c not in ['input_ids', 'attention_mask', 'labels']]
train_dataset = train_dataset.remove_columns([c for c in columns_to_remove if c in train_dataset.column_names])
val_dataset = val_dataset.remove_columns([c for c in columns_to_remove if c in val_dataset.column_names])
test_dataset = test_dataset.remove_columns([c for c in columns_to_remove if c in test_dataset.column_names])

# Set format
train_dataset.set_format('torch')
val_dataset.set_format('torch')
test_dataset.set_format('torch')

print("Datasets prepared!")

In [ ]:
# ============================================================================
# CELL 7: METRICS
# ============================================================================

def compute_metrics(eval_pred, threshold: float = 0.5):
    """Compute detailed metrics"""
    predictions, labels = eval_pred

    # Apply sigmoid and threshold
    predictions = torch.sigmoid(torch.tensor(predictions)).numpy()
    predictions = np.where(predictions > threshold, 1, 0)

    # Overall metrics
    f1_micro = f1_score(labels, predictions, average='micro', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    results = {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
    }

    # Per-label metrics
    for i, label in enumerate(LABEL_COLUMNS):
        f1 = f1_score(labels[:, i], predictions[:, i], zero_division=0)
        results[f'f1_{label}'] = f1

    return results

print("Metrics function defined!")

In [ ]:
# ============================================================================
# CELL 8: INITIALIZE MODEL
# ============================================================================

# Create label indices
label_indices = {label: i for i, label in enumerate(LABEL_COLUMNS)}

# Create model config
model_config = ConfigurableHierarchicalConfig(
    base_model=CONFIG["base_model"],
    use_hierarchy=CONFIG["use_hierarchy"],
    gated_hierarchy=CONFIG["gated_hierarchy"],
    gate_threshold=CONFIG["gate_threshold"],
    hidden_size=CONFIG["hidden_size"],
    hierarchical_hidden_size=CONFIG["hierarchical_hidden_size"],
    num_hidden_layers=CONFIG["num_hidden_layers"],
    dropout=CONFIG["dropout"],
    attention_dropout=CONFIG["attention_dropout"],
    use_weighted_loss=CONFIG["use_weighted_loss"],
    use_focal_loss=CONFIG["use_focal_loss"],
    focal_gamma=CONFIG["focal_gamma"],
    teacher_forcing_ratio=CONFIG["teacher_forcing_ratio"],
    predict_main_labels=False,
    num_main_labels=0,
    num_event_labels=3,  # Illness, Accident, Other
    num_cause_labels=4,  # Material_Physical, Spirits_Gods, Witchcraft_Sorcery, Rule_Violation_Taboo
    num_action_labels=5,  # Physical_Material, Technical_Specialist, Divination, Shaman_Medium_Healer, Priest_High_Religion
    total_labels=len(LABEL_COLUMNS),
    label_indices=label_indices,
    label_names=LABEL_COLUMNS,
)

# Initialize model
model = ConfigurableHierarchicalModel(model_config)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model initialized!")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

In [ ]:
# ============================================================================
# CELL 9: TRAINING
# ============================================================================

# Create output directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"models/training_{timestamp}"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

# Training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    learning_rate=CONFIG["learning_rate"],
    logging_dir=f'{output_dir}/logs',
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_micro",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    label_smoothing_factor=CONFIG["label_smoothing"],
    remove_unused_columns=False,
    use_mps_device=(device.type == 'mps'),
)

# Callbacks
callbacks = []
if CONFIG["use_early_stopping"]:
    callbacks.append(EarlyStoppingCallback(
        early_stopping_patience=CONFIG["early_stopping_patience"],
        early_stopping_threshold=CONFIG["early_stopping_threshold"]
    ))

# Initialize trainer
trainer = HierarchicalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    pos_weights=pos_weights if CONFIG["use_weighted_loss"] else None,
    neg_weights=neg_weights if CONFIG["use_weighted_loss"] else None,
    teacher_forcing_ratio=CONFIG["teacher_forcing_ratio"],
    callbacks=callbacks,
)

print(f"\nTrainer configured!")
print(f"  Training for {CONFIG['num_epochs']} epochs")
print(f"  Early stopping: {CONFIG['use_early_stopping']}")

# Train!
print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)

trainer.train()

print("\nTraining completed!")

In [ ]:
# ============================================================================
# CELL 10: EVALUATE ON TEST SET
# ============================================================================

print("=" * 60)
print("EVALUATING ON TEST SET")
print("=" * 60)

test_results = trainer.evaluate(eval_dataset=test_dataset)

print(f"\nTest Results:")
print(f"  F1 Micro: {test_results['eval_f1_micro']:.4f}")
print(f"  F1 Macro: {test_results['eval_f1_macro']:.4f}")

# Per-label results
print("\nPer-label F1:")
label_f1s = [(k.replace('eval_f1_', ''), v) for k, v in test_results.items() 
             if k.startswith('eval_f1_') and k not in ['eval_f1_micro', 'eval_f1_macro']]
label_f1s.sort(key=lambda x: x[1], reverse=True)

for label, f1 in label_f1s:
    bar = '█' * int(f1 * 20) + '░' * (20 - int(f1 * 20))
    print(f"  {label:25s} {bar} {f1:.3f}")

In [ ]:
# ============================================================================
# CELL 11: FIND OPTIMAL THRESHOLDS
# ============================================================================

def find_optimal_thresholds(model, dataset, label_names, thresholds=np.arange(0.3, 0.75, 0.05)):
    """Find optimal threshold per label"""
    print("Finding optimal thresholds...")
    
    predictions = trainer.predict(dataset)
    logits = predictions.predictions
    labels = predictions.label_ids
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    
    optimal_thresholds = {}
    
    for i, label_name in enumerate(label_names):
        best_f1 = 0
        best_threshold = 0.5
        
        for threshold in thresholds:
            preds = np.where(probs[:, i] > threshold, 1, 0)
            f1 = f1_score(labels[:, i], preds, zero_division=0)
            
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
        
        optimal_thresholds[label_name] = {
            'threshold': float(best_threshold),
            'f1_at_threshold': float(best_f1)
        }
        print(f"  {label_name}: threshold={best_threshold:.2f}, F1={best_f1:.3f}")
    
    return optimal_thresholds

optimal_thresholds = find_optimal_thresholds(model, test_dataset, LABEL_COLUMNS)

In [ ]:
# ============================================================================
# CELL 12: SAVE MODEL
# ============================================================================

final_model_path = f"{output_dir}/final_model"

# Save model and tokenizer
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

# Save training info
training_info = {
    "config": CONFIG,
    "test_results": {k: float(v) if isinstance(v, (np.floating, float)) else v 
                     for k, v in test_results.items()},
    "optimal_thresholds": optimal_thresholds,
    "label_columns": LABEL_COLUMNS,
    "model_info": {
        "total_params": total_params,
        "trainable_params": trainable_params
    },
    "training_completed": datetime.now().isoformat(),
    "dataset_size": {
        "train": len(train_df),
        "val": len(val_df),
        "test": len(test_df)
    }
}

with open(f"{final_model_path}/training_info.json", "w") as f:
    json.dump(training_info, f, indent=2)

print(f"Model saved to: {final_model_path}")
print(f"\nTo load this model:")
print(f"  model = AutoModel.from_pretrained('{final_model_path}')")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{final_model_path}')")

In [ ]:
# ============================================================================
# CELL 13: QUICK INFERENCE TEST
# ============================================================================

def predict_passage(text, model, tokenizer, label_names, thresholds=None):
    """Predict labels for a passage"""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
    
    predictions = {}
    for i, label in enumerate(label_names):
        threshold = thresholds.get(label, {}).get('threshold', 0.5) if thresholds else 0.5
        predictions[label] = {
            'probability': float(probs[i]),
            'predicted': probs[i] > threshold
        }
    
    return predictions

# Test
test_text = "The shaman performed a healing ritual to cure the illness caused by evil spirits."
print(f"Test text: '{test_text}'")
print("\nPredictions:")

preds = predict_passage(test_text, model, tokenizer, LABEL_COLUMNS, optimal_thresholds)
for label, info in preds.items():
    if info['predicted']:
        print(f"  ✓ {label}: {info['probability']:.3f}")